In [1]:
import pandas as pd
import joblib
import json
import warnings
import numpy as np
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# Load saved artifacts
# ---------------------------------------------------------------
model = joblib.load("attrition_model.pkl")

with open("threshold.json", "r") as f:
    threshold = json.load(f)["threshold"]

with open("columns.json", "r") as f:
    columns = json.load(f)

# Extract model internals for contribution scoring
preprocessor   = model.named_steps["preprocessor"]
logistic_model = model.named_steps["model"]
FEATURE_NAMES  = list(preprocessor.get_feature_names_out())
COEFS          = logistic_model.coef_[0]

# ---------------------------------------------------------------
# Human-readable labels for every feature the model uses
# ---------------------------------------------------------------
LABEL_MAP = {
    # --- Numeric / engineered features ---
    "standardscaler__WorkPressure":            "High overtime / poor work-life balance",
    "standardscaler__CommuteStress":           "High commute stress with overtime",
    "standardscaler__PromotionDelay":          "Promotion delay",
    "standardscaler__SalaryGrowth":            "Low salary growth",
    "standardscaler__CareerStagnation":        "Career stagnation",
    "standardscaler__Stability":               "Long company tenure",
    "standardscaler__IncomePerLevel":          "Low income for job level",
    "standardscaler__ManagerFeedback":         "Low manager feedback",
    "standardscaler__JobSatisfaction":         "Low job satisfaction",
    "standardscaler__EnvironmentSatisfaction": "Poor environment satisfaction",
    "standardscaler__StockOptionLevel":        "No / low stock options",
    "standardscaler__JobInvolvement":          "Low job involvement",
    "standardscaler__LeaveFrequency":          "High leave frequency",
    "standardscaler__Age":                     "Younger age (higher mobility)",
    "standardscaler__YearsWithCurrManager":    "Short time with current manager",
    "standardscaler__NumCompaniesWorked":      "Worked at many companies",
    "standardscaler__TrainingTimesLastYear":   "Low training opportunities",
    "standardscaler__RelationshipSatisfaction":"Low relationship satisfaction",
    "standardscaler__DailyRate":               "Low daily rate",
    "standardscaler__HourlyRate":              "Low hourly rate",
    "standardscaler__MonthlyRate":             "Low monthly rate",
    "standardscaler__Education":               "Lower education level",
    "standardscaler__PerformanceRating":       "Low performance rating",
    # --- One-hot: Business Travel ---
    "onehotencoder__BusinessTravel_Travel_Frequently": "Frequent business travel",
    "onehotencoder__BusinessTravel_Travel_Rarely":     "Rarely travels for business",
    "onehotencoder__BusinessTravel_Non-Travel":        "No business travel",
    # --- One-hot: Department ---
    "onehotencoder__Department_Sales":                 "Sales department",
    "onehotencoder__Department_Human Resources":       "HR department",
    "onehotencoder__Department_Research & Development":"R&D department",
    # --- One-hot: Education Field ---
    "onehotencoder__EducationField_Human Resources":   "HR education background",
    "onehotencoder__EducationField_Life Sciences":     "Life Sciences background",
    "onehotencoder__EducationField_Marketing":         "Marketing background",
    "onehotencoder__EducationField_Medical":           "Medical background",
    "onehotencoder__EducationField_Other":             "Other education background",
    "onehotencoder__EducationField_Technical Degree":  "Technical Degree background",
    # --- One-hot: Gender ---
    "onehotencoder__Gender_Male":                      "Male gender",
    "onehotencoder__Gender_Female":                    "Female gender",
    # --- One-hot: Job Role ---
    "onehotencoder__JobRole_Healthcare Representative":"Healthcare Representative role",
    "onehotencoder__JobRole_Human Resources":          "HR role",
    "onehotencoder__JobRole_Laboratory Technician":    "Lab Technician role",
    "onehotencoder__JobRole_Manager":                  "Manager role",
    "onehotencoder__JobRole_Manufacturing Director":   "Manufacturing Director role",
    "onehotencoder__JobRole_Research Director":        "Research Director role",
    "onehotencoder__JobRole_Research Scientist":       "Research Scientist role",
    "onehotencoder__JobRole_Sales Executive":          "Sales Executive role",
    "onehotencoder__JobRole_Sales Representative":     "Sales Representative role",
    # --- One-hot: Marital Status ---
    "onehotencoder__MaritalStatus_Single":             "Single (higher mobility)",
    "onehotencoder__MaritalStatus_Married":            "Married (stable)",
    "onehotencoder__MaritalStatus_Divorced":           "Divorced",
}

# ---------------------------------------------------------------
# Suggested actions mapped to each risk label
# ---------------------------------------------------------------
ACTION_MAP = {
    "High overtime / poor work-life balance":   "Discuss workload redistribution and offer flexible working hours",
    "High commute stress with overtime":         "Explore remote work or hybrid arrangements to reduce commute burden",
    "Promotion delay":                           "Schedule a career growth discussion and define a promotion timeline",
    "Low salary growth":                         "Review compensation and benchmark against market rates",
    "Career stagnation":                         "Offer new responsibilities, lateral moves, or upskilling programs",
    "Low manager feedback":                      "Facilitate regular 1:1s and manager-employee alignment sessions",
    "Low job satisfaction":                      "Conduct a stay interview to understand pain points and adjust role",
    "Poor environment satisfaction":             "Investigate team dynamics and improve workplace conditions",
    "No / low stock options":                    "Consider adding stock options or a long-term incentive plan",
    "Frequent business travel":                  "Review travel load and explore remote-work alternatives",
    "Single (higher mobility)":                  "Strengthen engagement through mentorship and team bonding",
    "Short time with current manager":           "Encourage manager to build rapport through structured check-ins",
    "High leave frequency":                      "Conduct a proactive stay interview to address underlying concerns",
    "Worked at many companies":                  "Offer role stability signals such as long-term projects or promotions",
    "Low job involvement":                       "Assign high-impact projects to increase ownership and engagement",
    "Low training opportunities":                "Enroll employee in relevant training or certification programs",
    "Younger age (higher mobility)":             "Invest in career development plans to build long-term commitment",
    "Low income for job level":                  "Review pay-grade alignment and address any compensation gaps",
}

DEFAULT_ACTION = "Continue regular check-ins and maintain current engagement practices"

# ---------------------------------------------------------------
# Feature Engineering — must exactly match training
# ---------------------------------------------------------------
def engineer_features(df):
    df = df.copy()
    df["SalaryGrowth"]     = df["PercentSalaryHike"] / (df["YearsAtCompany"] + 1)
    df["PromotionDelay"]   = df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)
    df["WorkPressure"]     = df["OverTime"] * (4 - df["WorkLifeBalance"])
    df["CareerStagnation"] = (df["YearsInCurrentRole"] + df["YearsSinceLastPromotion"]) / (df["TotalWorkingYears"] + 1)
    df["Stability"]        = df["YearsAtCompany"] / (df["TotalWorkingYears"] + 1)
    df["IncomePerLevel"]   = df["MonthlyIncome"] / (df["JobLevel"] + 1)
    df["CommuteStress"]    = df["DistanceFromHome"] * df["OverTime"]
    df.drop([
        "PercentSalaryHike", "YearsAtCompany", "YearsSinceLastPromotion",
        "OverTime", "WorkLifeBalance", "YearsInCurrentRole",
        "TotalWorkingYears", "MonthlyIncome", "JobLevel", "DistanceFromHome"
    ], axis=1, inplace=True)
    return df

# ---------------------------------------------------------------
# Model-Driven Contribution Scoring
#
# How it works:
#   1. Pass input through the pipeline's preprocessor (OHE + scaling)
#   2. Multiply each transformed value by its logistic regression coefficient
#   3. contribution_i = transformed_value_i * coef_i
#      → Positive contribution = pushes model toward LEAVE
#      → Negative contribution = pushes model toward STAY
#   4. Sort by contribution to get ranked risk and protective factors
#
# This is genuinely model-driven — the reasons come directly from
# what the model weighted, not hand-written if/else rules.
# ---------------------------------------------------------------
def get_contributions(df_aligned):
    X_transformed = preprocessor.transform(df_aligned)
    contributions = X_transformed[0] * COEFS          # shape: (49,)

    contrib_pairs = list(zip(FEATURE_NAMES, contributions))

    # Risk drivers: positive contribution (push toward LEAVE), ignore near-zero noise
    risk = [(LABEL_MAP.get(f, f), round(c, 4))
            for f, c in contrib_pairs if c > 0.05]
    risk.sort(key=lambda x: -x[1])

    # Protective factors: negative contribution (push toward STAY)
    protect = [(LABEL_MAP.get(f, f), round(abs(c), 4))
               for f, c in contrib_pairs if c < -0.05]
    protect.sort(key=lambda x: -x[1])

    return risk, protect

# ---------------------------------------------------------------
# Main Prediction Function
# ---------------------------------------------------------------
def predict(sample, name="Employee"):
    df      = pd.DataFrame([sample])
    df_eng  = engineer_features(df)
    df_ali  = df_eng.reindex(columns=columns, fill_value=0)

    prob  = model.predict_proba(df_ali)[:, 1][0]
    pred  = int(prob >= threshold)

    risk_drivers, protect_factors = get_contributions(df_ali)

    # Top 2 risk drivers become the reason string
    if pred == 1:
        top_reasons = [r[0] for r in risk_drivers[:2]]
        reason_str  = " + ".join(top_reasons) if top_reasons else "Multiple combined risk signals"
    else:
        top_reasons = [p[0] for p in protect_factors[:2]]
        reason_str  = " + ".join(top_reasons) if top_reasons else "Low overall risk profile"

    # Suggested action from top risk driver
    action = DEFAULT_ACTION
    if pred == 1 and risk_drivers:
        a1 = ACTION_MAP.get(risk_drivers[0][0], DEFAULT_ACTION)
        action = a1
        if len(risk_drivers) > 1:
            a2 = ACTION_MAP.get(risk_drivers[1][0])
            if a2 and a2 != a1:
                action = a1 + "\n" + " " * 21 + a2

    # ---- Print output ----
    bar = "=" * 58
    print(f"\n{bar}")
    print(f"  Employee         : {name}")
    print(f"{bar}")
    print(f"  Attrition Risk   : {round(prob * 100, 1)}%")
    print(f"  Prediction       : {'⚠️  LIKELY TO LEAVE' if pred == 1 else '✅  LIKELY TO STAY'}")
    print(f"  Reason           : {reason_str}")
    print(f"  Suggested Action : {action}")

    print(f"\n  {'📊 Model-Driven Risk Drivers (contribution score)':}")
    if risk_drivers:
        for label, score in risk_drivers[:5]:
            bar_len = int(score * 20)
            print(f"    [+{score:.3f}] {'█' * bar_len} {label}")
    else:
        print("    None above threshold")

    print(f"\n  {'🛡️  Protective Factors (contribution score)':}")
    if protect_factors:
        for label, score in protect_factors[:5]:
            bar_len = int(score * 20)
            print(f"    [-{score:.3f}] {'░' * bar_len} {label}")
    else:
        print("    None above threshold")

    print(f"{'=' * 58}\n")

# ================================================================
# Test Cases
# ================================================================

# Case 1: Very high risk — Sales Executive, single, overtime, low pay
predict({
    "BusinessTravel":"Travel_Frequently","Department":"Sales","EducationField":"Marketing",
    "Gender":"Male","JobRole":"Sales Executive","MaritalStatus":"Single",
    "Age":26,"DailyRate":400,"Education":2,"EnvironmentSatisfaction":1,"HourlyRate":45,
    "JobInvolvement":2,"JobSatisfaction":1,"MonthlyRate":9000,"NumCompaniesWorked":3,
    "PerformanceRating":3,"RelationshipSatisfaction":2,"StockOptionLevel":0,
    "TrainingTimesLastYear":1,"YearsWithCurrManager":0,"LeaveFrequency":0.15,"ManagerFeedback":0.40,
    "PercentSalaryHike":10,"YearsAtCompany":1,"YearsSinceLastPromotion":1,"OverTime":1,
    "WorkLifeBalance":1,"YearsInCurrentRole":0,"TotalWorkingYears":2,"MonthlyIncome":2000,
    "JobLevel":1,"DistanceFromHome":25,
}, name="Arjun Singh  |  Sales Executive")

# Case 2: Very low risk — Senior researcher, married, high pay, long tenure
predict({
    "BusinessTravel":"Travel_Rarely","Department":"Research & Development","EducationField":"Life Sciences",
    "Gender":"Female","JobRole":"Research Scientist","MaritalStatus":"Married",
    "Age":38,"DailyRate":900,"Education":3,"EnvironmentSatisfaction":4,"HourlyRate":70,
    "JobInvolvement":3,"JobSatisfaction":4,"MonthlyRate":15000,"NumCompaniesWorked":2,
    "PerformanceRating":3,"RelationshipSatisfaction":4,"StockOptionLevel":2,
    "TrainingTimesLastYear":3,"YearsWithCurrManager":6,"LeaveFrequency":0.02,"ManagerFeedback":0.85,
    "PercentSalaryHike":20,"YearsAtCompany":10,"YearsSinceLastPromotion":1,"OverTime":0,
    "WorkLifeBalance":3,"YearsInCurrentRole":5,"TotalWorkingYears":12,"MonthlyIncome":15000,
    "JobLevel":3,"DistanceFromHome":5,
}, name="Priya Mehta  |  Research Scientist")

# Case 3: Medium-high risk — Manager with overtime, poor satisfaction, stuck in role
predict({
    "BusinessTravel":"Travel_Rarely","Department":"Research & Development","EducationField":"Medical",
    "Gender":"Male","JobRole":"Manager","MaritalStatus":"Married",
    "Age":45,"DailyRate":750,"Education":4,"EnvironmentSatisfaction":2,"HourlyRate":60,
    "JobInvolvement":2,"JobSatisfaction":2,"MonthlyRate":13000,"NumCompaniesWorked":5,
    "PerformanceRating":3,"RelationshipSatisfaction":2,"StockOptionLevel":1,
    "TrainingTimesLastYear":2,"YearsWithCurrManager":1,"LeaveFrequency":0.10,"ManagerFeedback":0.45,
    "PercentSalaryHike":12,"YearsAtCompany":3,"YearsSinceLastPromotion":3,"OverTime":1,
    "WorkLifeBalance":2,"YearsInCurrentRole":2,"TotalWorkingYears":15,"MonthlyIncome":8000,
    "JobLevel":2,"DistanceFromHome":10,
}, name="Vikram Nair  |  Manager")

# Case 4: Borderline — HR professional, average scores across the board
predict({
    "BusinessTravel":"Travel_Rarely","Department":"Human Resources","EducationField":"Human Resources",
    "Gender":"Female","JobRole":"Human Resources","MaritalStatus":"Divorced",
    "Age":32,"DailyRate":600,"Education":3,"EnvironmentSatisfaction":3,"HourlyRate":55,
    "JobInvolvement":3,"JobSatisfaction":3,"MonthlyRate":11000,"NumCompaniesWorked":3,
    "PerformanceRating":3,"RelationshipSatisfaction":3,"StockOptionLevel":0,
    "TrainingTimesLastYear":2,"YearsWithCurrManager":2,"LeaveFrequency":0.07,"ManagerFeedback":0.60,
    "PercentSalaryHike":13,"YearsAtCompany":4,"YearsSinceLastPromotion":2,"OverTime":0,
    "WorkLifeBalance":3,"YearsInCurrentRole":3,"TotalWorkingYears":8,"MonthlyIncome":5000,
    "JobLevel":2,"DistanceFromHome":15,
}, name="Sneha Rao    |  HR Specialist")

# Case 5: Very low risk — Senior Director, long tenure, no overtime, top stock options
predict({
    "BusinessTravel":"Non-Travel","Department":"Research & Development","EducationField":"Life Sciences",
    "Gender":"Male","JobRole":"Research Director","MaritalStatus":"Married",
    "Age":52,"DailyRate":1200,"Education":5,"EnvironmentSatisfaction":4,"HourlyRate":90,
    "JobInvolvement":4,"JobSatisfaction":4,"MonthlyRate":18000,"NumCompaniesWorked":1,
    "PerformanceRating":4,"RelationshipSatisfaction":4,"StockOptionLevel":3,
    "TrainingTimesLastYear":4,"YearsWithCurrManager":8,"LeaveFrequency":0.01,"ManagerFeedback":0.92,
    "PercentSalaryHike":18,"YearsAtCompany":20,"YearsSinceLastPromotion":2,"OverTime":0,
    "WorkLifeBalance":4,"YearsInCurrentRole":8,"TotalWorkingYears":28,"MonthlyIncome":18000,
    "JobLevel":5,"DistanceFromHome":3,
}, name="Rohit Sharma |  Research Director")


  Employee         : Arjun Singh  |  Sales Executive
  Attrition Risk   : 99.5%
  Prediction       : ⚠️  LIKELY TO LEAVE
  Reason           : High overtime / poor work-life balance + High commute stress with overtime
  Suggested Action : Discuss workload redistribution and offer flexible working hours
                     Explore remote work or hybrid arrangements to reduce commute burden

  📊 Model-Driven Risk Drivers (contribution score)
    [+1.352] ███████████████████████████ High overtime / poor work-life balance
    [+0.945] ██████████████████ High commute stress with overtime
    [+0.423] ████████ Low manager feedback
    [+0.405] ████████ Low income for job level
    [+0.385] ███████ Younger age (higher mobility)

  🛡️  Protective Factors (contribution score)
    [-0.077] ░ Long company tenure
    [-0.063] ░ Lower education level


  Employee         : Priya Mehta  |  Research Scientist
  Attrition Risk   : 1.4%
  Prediction       : ✅  LIKELY TO STAY
  Reason           : Low i